In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import jacfwd
from jax import jit
from jax import vmap
from functools import partial
import sys
sys.path.append("../src/gapmoe/")
import calc_vEarth
import EarthMotion

In [2]:
tref = 10063.874
RA_str, Dec_str = "17:57:38.03","-28:38:28.53"
RA_deg = calc_vEarth.hms_string_to_degrees(RA_str)
Dec_deg = calc_vEarth.dms_string_to_degrees(Dec_str)
vEarth = np.array(calc_vEarth.calc_vSun(tref,RA_deg,Dec_deg))
print(vEarth)
vEarth2 = EarthMotion.calc_vEarth(2450000+tref,RA_deg,Dec_deg)
print(vEarth2)

[-0.5661249   0.61675232]
(np.float64(-0.42944823204271027), np.float64(-3.854637062559252))


In [3]:
@jit
def physical_to_lightcurve_circular_old(theta,thS):
    tE     = theta[1]
    rho    = theta[3]
    s      = theta[5]
    piEN   = theta[7]
    piEE   = theta[8]
    gamma1 = theta[9]
    gamma2 = theta[10]
    gamma3 = theta[11]

    piE = jnp.sqrt(piEN**2 + piEE**2)

    thE = thS / rho
    ML = thE / 8.1439 / piE # KAPPA = 8.1429 [mas / Msun]
    murel = thE / tE * 365.25
    murel_E = murel * piEE / piE
    murel_N = murel * piEN / piE

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * 2.959122082855911e-4)) # G = 2.959122082855911e-4 [AU^3 / (Msun * day^2)]
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE)

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L

    RE = DL * thE
    orbital_radi = RE * s * gamma_ratio

    cosi = gamma2 / (gamma_ratio * gamma_abs)
    tanphi = - gamma1 * gamma_abs / (gamma3 * gamma_parallel)

    return  jnp.array([murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi])

@jit
def wrapped_physical_to_lightcurve_circular_old(theta_reduced, thS):
    tE, rho, s, piEN, piEE, gamma1, gamma2, gamma3 = theta_reduced

    piE = jnp.sqrt(piEE**2 + piEN**2)
    thE = thS / rho
    ML = thE / 8.1439 / piE
    murel = thE / tE * 365.25
    murel_E = murel * piEE / piE
    murel_N = murel * piEN / piE

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * 2.959122082855911e-4))
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE)

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L

    RE = DL * thE
    orbital_radi = RE * s * gamma_ratio

    cosi = gamma2 / (gamma_ratio * gamma_abs)
    tanphi = - gamma1 * gamma_abs / (gamma3 * gamma_parallel)

    return jnp.array([murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi])

@jit
def calc_ln_det_jacobian_circular_old(theta, thS):
    theta_reduced = jnp.array([
        theta[1],
        theta[3],
        theta[5],
        theta[7],
        theta[8],
        theta[9],
        theta[10],
        theta[11]
    ])
    J = jacfwd(wrapped_physical_to_lightcurve_circular_old)(theta_reduced, thS)
    sign, lndet = jnp.linalg.slogdet(J)
    return lndet

In [4]:
t0 = 10090 - 5       
u0 = 0.01          
q = 0.001
alpha = np.pi/2 + 1
tE = 47.1147
rho = 0.000528216
s = 1.22626
piEN = 0.0565383
piEE = 0.0565382
gamma1 = 0.000702119
gamma2 = 0.00175531
gamma3 = 0.0017553
thS = 0.0001800539

In [5]:
x =  jnp.array([t0, tE, u0, rho, q, s, alpha, piEN, piEE, gamma1, gamma2, gamma3])
murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi = physical_to_lightcurve_circular_old(x, thS)

print(f"μ_rel_E   = {murel_E:.6f}  # mas/yr")
print(f"μ_rel_N   = {murel_N:.6f}  # mas/yr")
print(f"M_L       = {ML:.6f}       # M_sun")
print(f"D_L       = {DL:.6f}       # kpc")
print(f"D_S       = {Ds:.6f}       # kpc")
print(f"R_orbit   = {orbital_radi:.6f}  # AU")
print(f"cos(i)    = {cosi:.6f}")
print(f"tan(φ)    = {tanphi:.6f}")

print(calc_ln_det_jacobian_circular_old(x, thS))

μ_rel_E   = 1.868570  # mas/yr
μ_rel_N   = 1.868573  # mas/yr
M_L       = 0.523481       # M_sun
D_L       = 6.663750       # kpc
D_S       = 8.142625       # kpc
R_orbit   = 2.999999  # AU
cos(i)    = 0.631751
tan(φ)    = -0.545831
32.090496


In [6]:
@jit
def physical_to_lightcurve_circular(theta,thS,vEarth):
    tE     = theta[1]
    rho    = theta[3]
    s      = theta[5]
    piEN   = theta[7]
    piEE   = theta[8]
    gamma1 = theta[9]
    gamma2 = theta[10]
    gamma3 = theta[11]
    vEarth_N, vEarthE = vEarth

    piE = jnp.sqrt(piEN**2 + piEE**2)

    thE = thS / rho
    ML = thE / 8.1439 / piE # KAPPA = 8.1429 [mas / Msun]
    murel_geo = thE / tE * 365.25
    murel_E_geo = murel_geo * piEE / piE
    murel_N_geo = murel_geo * piEN / piE

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * 2.959122082855911e-4)) # G = 2.959122082855911e-4 [AU^3 / (Msun * day^2)]
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE)

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L

    RE = DL * thE
    orbital_radi = RE * s * gamma_ratio

    cosi = gamma2 / (gamma_ratio * gamma_abs)
    tanphi = - gamma1 * gamma_abs / (gamma3 * gamma_parallel)

    return  jnp.array([murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi])

In [7]:
physical_to_lightcurve_circular(x,thS,vEarth)

Array([ 1.8685734 ,  1.8685701 ,  0.52348137,  6.6637497 ,  8.142625  ,
        2.9999993 ,  0.6317505 , -0.5458313 ], dtype=float32)

In [8]:
import numpy as np
from astropy.time import Time
from astropy.coordinates import get_body_barycentric_posvel, SkyCoord
from astropy.coordinates import ICRS, CartesianRepresentation
from astropy import units as u
from astropy.coordinates import solar_system_ephemeris

def calc_vEarth_test(jd, ra_deg, dec_deg):
    # --- 観測時刻と方向（SkyCoord）を作成 ---
    time = Time(jd, format='jd', scale='tdb')
    target = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame='icrs')

    # --- 地球の位置と速度（太陽中心）を取得 ---
    with solar_system_ephemeris.set('de432s'):
        pos, vel = get_body_barycentric_posvel('earth', time)

    # vel: barycentric velocity in AU/day (CartesianRepresentation)

    # --- 単位ベクトルを構成（RA/Decベース） ---
    # 視線方向（LOS）ベクトル
    los = target.cartesian.xyz.value
    los /= np.linalg.norm(los)

    # 赤緯方向（北：+Dec）単位ベクトル
    d_dec = SkyCoord(ra=ra_deg * u.deg, dec=(dec_deg + 1e-5) * u.deg, frame='icrs')
    north = d_dec.cartesian.xyz.value - target.cartesian.xyz.value
    north /= np.linalg.norm(north)

    # 赤経方向（東：+RA）単位ベクトル
    d_ra = SkyCoord(ra=(ra_deg + 1e-5) * u.deg, dec=dec_deg * u.deg, frame='icrs')
    east = d_ra.cartesian.xyz.value - target.cartesian.xyz.value
    east /= np.linalg.norm(east)

    # --- 速度を投影（AU/day）---
    v_vec = vel.xyz.to_value(u.AU / u.year)  # ndarray (3,)

    v_north = np.dot(v_vec, north)  # 北方向成分
    v_east = np.dot(v_vec, east)    # 東方向成分

    return v_north, v_east

In [9]:
calc_vEarth_test(tref+2450000,RA_deg,Dec_deg)

(np.float64(0.42962597852264967), np.float64(3.8570348691472542))

In [10]:
import numpy as np

def calc_vEarth_new(t_jd, ra_deg, dec_deg):
    # === JPL要素（C++と同じ） ===
    a0 = 1.00000261
    adot = 0.00000562
    e0 = 0.01671123
    edot = -0.00004392
    inc0 = -0.00001531
    incdot = -0.01294668
    L0 = 100.46457166
    Ldot = 35999.37244981
    om0 = 102.93768193
    omdot = 0.32327364
    eps0 = 23.439281
    deg = np.pi / 180

    # === 時刻補正（ユリウス世紀） ===
    T = (t_jd - 2451545.0) / 36525.0

    # === 軌道要素 ===
    a = a0 + adot * T
    e = e0 + edot * T
    inc = (inc0 + incdot * T) * deg
    L = (L0 + Ldot * T) * deg
    om = (om0 + omdot * T) * deg
    eps = eps0 * deg
    M = L - om
    M %= 2 * np.pi

    # === Kepler方程式を解く ===
    E = M  # 初期推定
    for _ in range(100):
        dE = (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        E -= dE
        if abs(dE) < 1e-10:
            break

    # === 地球の位置（AU）と速度（AU/day）===
    cosE, sinE = np.cos(E), np.sin(E)
    r = a * (1 - e * cosE)
    x = a * (cosE - e)
    y = a * np.sqrt(1 - e**2) * sinE

    # === 公転速度（平面上）===
    n = Ldot * deg / 36525  # 平均運動 [rad/day]
    vx = -a * sinE * n / (1 - e * cosE)
    vy = a * np.sqrt(1 - e**2) * cosE * n / (1 - e * cosE)

    # === 3次元変換（傾斜・近日点） ===
    cos_om = np.cos(om)
    sin_om = np.sin(om)
    cos_inc = np.cos(inc)
    sin_inc = np.sin(inc)

    # 地球の位置（不要だが残す）
    ex = x * cos_om - y * sin_om
    ey = x * sin_om * cos_inc + y * cos_om * cos_inc
    ez = x * sin_om * sin_inc + y * cos_om * sin_inc

    # 地球の速度ベクトル
    vx3 = vx * cos_om - vy * sin_om
    vy3 = vx * sin_om * cos_inc + vy * cos_om * cos_inc
    vz3 = vx * sin_om * sin_inc + vy * cos_om * sin_inc

    # 地球から見た太陽の速度（逆向き）
    vsun_ec = -np.array([vx3, vy3, vz3]) # 黄道座標系 (ecliptic)
    
    R = np.array([
    [1, 0, 0],
    [0,  np.cos(eps), -np.sin(eps)],
    [0,  np.sin(eps),  np.cos(eps)]
    ])
    vsun_eq = R @ vsun_ec

    # === RA/Dec方向から LOS・東・北ベクトルを構成 ===
    ra = ra_deg * deg
    dec = dec_deg * deg
    
    earth_north = np.array([0,0,1])

    los = np.array([
        np.cos(dec) * np.cos(ra),
        np.cos(dec) * np.sin(ra),
        np.sin(dec)
    ])
    
    east = np.cross(earth_north, los)
    east /= np.sqrt(np.dot(east, east))
    north = np.cross(los, east)            

    # === 投影 ===
    v_north = np.dot(vsun_eq, north)
    v_east  = np.dot(vsun_eq, east)

    return v_north * 365.25, v_east*365.25

In [20]:
tref = 4000
RA_str, Dec_str = "19:57:38.03","-28:38:28.53"
RA_deg = calc_vEarth.hms_string_to_degrees(RA_str)
Dec_deg = calc_vEarth.dms_string_to_degrees(Dec_str)

print(EarthMotion.calc_vEarth(2450000+tref,RA_deg,Dec_deg))
print(calc_vEarth_new(2450000+tref,RA_deg,Dec_deg))
print(calc_vEarth_test(tref+2450000,RA_deg,Dec_deg))
print(np.array(calc_vEarth.calc_vSun(tref,RA_deg,Dec_deg)))

(np.float64(0.1924550325068491), np.float64(-2.8946064503032902))
(np.float64(0.1924550325068491), np.float64(-2.8946064503032902))
(np.float64(-0.19265899561842434), np.float64(2.895237714050396))
[-0.83241947 -5.82776952]
